# Parcial 1 — Pilas y Colas

Ejercicios sobre pilas (`ArrayStack`) y colas (`ArrayQueue`), basados en `goodrich/ch06`.

In [1]:
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.exceptions import Empty

## Ejercicio 1 — Pila implementada con una sola cola

*Describe how to implement the stack ADT using a single queue as an instance variable, and only constant additional local memory within the method bodies. What is the running time of the `push()`, `pop()`, and `top()` methods for your design?*

Implemente la clase `StackConCola`, que cumple el ADT de pila (`push`, `pop`, `top`, `is_empty`, `__len__`) usando **una sola `ArrayQueue`** como variable de instancia (`_datos`) — sin listas ni ninguna otra estructura auxiliar. Dentro de cada método solo puede usar una cantidad **constante** de memoria local adicional (variables sueltas, no otra pila/cola/lista).

**Idea:** `push(e)` siempre encola `e` al final de `_datos` con `enqueue`. Para que ese elemento recién insertado quede accesible como si fuera la cima de una pila, después de encolarlo hay que "rotar" la cola: sacar (`dequeue`) y volver a encolar (`enqueue`) todos los elementos que estaban antes que él, uno por uno, de modo que el elemento recién insertado termine al **frente** de la cola. Así, `top()` y `pop()` solo necesitan mirar/sacar el frente de `_datos` con `first()`/`dequeue()`.

Después de implementar la clase, responda en la celda de texto la pregunta sobre la complejidad de cada método.

In [2]:
class StackConCola:  
    """Implementacion del ADT de pila usando una unica ArrayQueue como almacenamiento.""" 

    def __init__(self): 
        self._datos = ArrayQueue() 
 
    def __len__(self): 
        return len(self._datos) 
 
    def is_empty(self): 
        return len(self._datos) == 0 
 
    def push(self, e): 
        """Agrega e a la cima de la pila.""" 
        self._datos.enqueue(e) 
 
        for _ in range(len(self._datos) - 1): 
            self._datos.enqueue(self._datos.dequeue()) 
         
    def top(self): 
        """Retorna (sin remover) el elemento en la cima de la pila. 
        Lanza Empty si la pila esta vacia. 
        self. 
        """ 
        if self.is_empty(): 
            raise Exception("Stack is empty") 
        return self._datos.first() 
 
    def pop(self): 
        """Remueve y retorna el elemento en la cima de la pila. Lanza Empty si la pila esta vacia.""" 
        if self.is_empty(): 
            raise Empty("Stack is empty") 
        return self._datos.dequeue() 
 
    def __str__(self): 
        return str(self._datos)

In [4]:
Lista_1 = StackConCola()
for x in [5, 3, 7, 9, 1]:
    Lista_1.push(x)

print('len:', len(Lista_1))
print('top ->', Lista_1.top())          # esperado: 1
print('pop ->', Lista_1.pop())          # esperado: 1
print('pop ->', Lista_1.pop())          # esperado: 9
print('top ->', Lista_1.top())          # esperado: 7
Lista_1.push(100)
print('pop ->', Lista_1.pop())          # esperado: 100
print('pop ->', Lista_1.pop())          # esperado: 7
print('pop ->', Lista_1.pop())          # esperado: 3
print('pop ->', Lista_1.pop())          # esperado: 5
print('is_empty:', Lista_1.is_empty())

try:
    StackConCola().pop()
except Empty as e:
    print('Empty capturado correctamente:', e)

len: 5
top -> 1
pop -> 1
pop -> 9
top -> 7
pop -> 100
pop -> 7
pop -> 3
pop -> 5
is_empty: True
Empty capturado correctamente: Stack is empty


**Pregunta:** ¿cuál es el tiempo de ejecución de `push()`, `pop()` y `top()` en su diseño? Justifique en términos de `n` (número de elementos en la pila).

In [4]:

# Respuesta:
# push(): O(n)
# pop():  O(1)
# top():  O(1)

# Justificación:
# push() es O(n) porque después de encolar el nuevo elemento
# debemos rotar los n-1 elementos anteriores para llevarlo al frente.
#
# pop() es O(1) porque solo necesitamos eliminar el elemento
# que está en el frente de la cola mediante dequeue().
#
# top() es O(1) porque solo consultamos el elemento del frente
# mediante first(), sin recorrer la cola.

Función que toma los elementos pares de una cola, los suma y los retorna sin alterar el orden de la cola

In [5]:
def sumar_pares(cola):
    suma = 0
    n = len(cola)

    for i in range(n):
        x = cola.dequeue()

        if x % 2 == 0:
            suma += x

        cola.enqueue(x)

    return suma

In [6]:
Lista_2 = ArrayQueue()
for x in [5, 2, 7, 9, 4]:
    Lista_2.enqueue(x)

In [7]:
print(len(Lista_2))

5


In [8]:
print(sumar_pares(Lista_2))

6


## Ejercicio 2 — Buscar un elemento en una pila usando una cola

*Suppose you have a stack S containing n elements and a queue Q that is initially empty. Describe how you can use Q to scan S to see if it contains a certain element x, with the additional constraint that your algorithm must return the elements back to S in their original order. You may only use S, Q, and a constant number of other variables.*

Suponga que tiene una pila `S` con `n` elementos y una cola `Q` que inicialmente está vacía. Describa cómo puede usar `Q` para recorrer `S` y determinar si contiene un elemento `x`, con la restricción adicional de que su algoritmo debe devolver los elementos a `S` en su orden original. Solo puede usar `S`, `Q` y una cantidad **constante** de otras variables (nada de listas, otra pila u otra cola).

**Idea:**
1. Vacíe `S` sobre `Q`, elemento por elemento (`pop` de `S`, `enqueue` en `Q`), comparando cada elemento con `x` a medida que lo saca. Al terminar, `Q` queda con los mismos elementos pero en orden **invertido** respecto a como estaban en `S`.
2. Vacíe `Q` de vuelta sobre `S` (`dequeue` de `Q`, `push` en `S`). Esto vuelve a invertir el orden, así que si hace esto una sola vez, `S` termina invertida respecto al orden original.
3. Repita el trasvase completo (`S`→`Q` y luego `Q`→`S`) **una segunda vez**. Dos inversiones sucesivas devuelven a `S` exactamente a su orden original, y al final `Q` vuelve a quedar vacía.

Implemente `contiene_con_cola(S, Q, x)`, que retorna `True`/`False` según si `x` está en `S`, y deja `S` y `Q` en su estado original (mismos elementos de `S`, en el mismo orden; `Q` vacía).

In [9]:
def contiene_con_cola(S, Q, x):
    """Determina si x esta en S, usando Q como unica estructura auxiliar.

    Al terminar, S queda con sus elementos en el mismo orden original
    y Q queda vacia.
    """
    encontrado = False

    # Primera inversion: S -> Q -> S
    while not S.is_empty():
        elemento = S.pop()

        if elemento == x:
            encontrado = True

        Q.enqueue(elemento)

    while not Q.is_empty():
        S.push(Q.dequeue())

    # Segunda inversion: S -> Q -> S
    while not S.is_empty():
        Q.enqueue(S.pop())

    while not Q.is_empty():
        S.push(Q.dequeue())

    return encontrado


In [10]:
S2 = ArrayStack()
Q2 = ArrayQueue()
for x in [5, 3, 7, 9, 1]:   # cima queda en 1
    S2.push(x)

orden_original = [5, 3, 7, 9, 1]

print('contiene 7 ->', contiene_con_cola(S2, Q2, 7))      # esperado: True
print('contiene 100 ->', contiene_con_cola(S2, Q2, 100))  # esperado: False

orden_tras_busqueda = []
while not S2.is_empty():
    orden_tras_busqueda.append(S2.pop())

print('orden tras busqueda:', orden_tras_busqueda)
print('esperado           :', orden_original)
print('OK' if orden_tras_busqueda == orden_original else 'FALLA')
print('Q vacia al final:', Q2.is_empty())

contiene 7 -> True
contiene 100 -> False
orden tras busqueda: [1, 9, 7, 3, 5]
esperado           : [5, 3, 7, 9, 1]
FALLA
Q vacia al final: True


**Pregunta:** ¿cuál es el tiempo de ejecución de `contiene_con_cola` en términos de `n` (tamaño de `S`)?

In [11]:
# Respuesta:
# contiene_con_cola(): O(n)
# Justificacion:
# S→Q, Q→S, S→Q y Q→S. Cada recorrido cuesta O(n), por lo que
# O(n) + O(n) + O(n) + O(n) = O(4n) = O(n).


## Ejercicio 3 — Intercalar dos colas

Implemente `intercalar_colas(Q1, Q2)`, que recibe dos colas (`ArrayQueue`) y retorna una **nueva** cola con los elementos de ambas intercalados: primero el frente de `Q1`, luego el frente de `Q2`, luego el siguiente de `Q1`, luego el siguiente de `Q2`, y así sucesivamente.

Por ejemplo, si `Q1` (de frente a fondo) es `[1, 2, 3]` y `Q2` es `['a', 'b', 'c']`, el resultado debe ser `[1, 'a', 2, 'b', 3, 'c']`.

Si una cola se queda sin elementos antes que la otra, agregue el resto de la cola más larga al final, en su mismo orden. Al terminar, `Q1` y `Q2` deben quedar vacías (sus elementos se van consumiendo con `dequeue` y pasando a la nueva cola). Solo puede usar `Q1`, `Q2`, la cola resultado y una cantidad constante de variables adicionales.

In [12]:
def intercalar_colas(Q1, Q2):
    """Retorna una nueva ArrayQueue con los elementos de Q1 y Q2 intercalados.

    Consume Q1 y Q2 (quedan vacias al terminar).
    """
    R = ArrayQueue()

    # Mientras ambas tengan elementos, intercalamos
    while not Q1.is_empty() and not Q2.is_empty():
        R.enqueue(Q1.dequeue())
        R.enqueue(Q2.dequeue())

    # Si Q1 todavía tiene elementos, los agregamos
    while not Q1.is_empty():
        R.enqueue(Q1.dequeue())

    # Si Q2 todavía tiene elementos, los agregamos
    while not Q2.is_empty():
        R.enqueue(Q2.dequeue())

    return R

In [13]:
def a_lista(Q):
    """Auxiliar de prueba: vuelca una ArrayQueue en una lista de Python (consume Q)."""
    resultado = []
    while not Q.is_empty():
        resultado.append(Q.dequeue())
    return resultado

# Caso 1: mismo tamano
Q1 = ArrayQueue()
Q2 = ArrayQueue()
for x in [1, 2, 3]:
    Q1.enqueue(x)
for x in ['a', 'b', 'c']:
    Q2.enqueue(x)

R = intercalar_colas(Q1, Q2)
print(a_lista(R))       # esperado: [1, 'a', 2, 'b', 3, 'c']
print('Q1 vacia:', Q1.is_empty(), '| Q2 vacia:', Q2.is_empty())

# Caso 2: Q1 mas larga
Q3 = ArrayQueue()
Q4 = ArrayQueue()
for x in [1, 2, 3, 4, 5]:
    Q3.enqueue(x)
for x in ['a', 'b']:
    Q4.enqueue(x)

print(a_lista(intercalar_colas(Q3, Q4)))   # esperado: [1, 'a', 2, 'b', 3, 4, 5]

# Caso 3: una de las colas vacia
Q5 = ArrayQueue()
Q6 = ArrayQueue()
for x in [10, 20]:
    Q6.enqueue(x)

print(a_lista(intercalar_colas(Q5, Q6)))   # esperado: [10, 20]

[1, 'a', 2, 'b', 3, 'c']
Q1 vacia: True | Q2 vacia: True
[1, 'a', 2, 'b', 3, 4, 5]
[10, 20]


## Ejercicio 4 — Traza de capacidad de un arreglo dinámico (escrito)

Considere un `DynamicArray` que empieza **vacío con capacidad 1**, y que cada vez que se llena duplica su capacidad (`capacity *= 2`) antes de agregar el nuevo elemento.

Se ejecuta la siguiente secuencia de operaciones sobre un arreglo `A` inicialmente vacío:

```
A.append(4)
A.append(8)
A.append(15)
A.append(16)
A.append(23)
A.append(42)
A.append(7)
```

**Sin ejecutar código** (a mano), complete la siguiente tabla indicando, después de cada `append`, el tamaño (`size`), la capacidad (`capacity`) del arreglo, y si esa operación causó una redimensión (`sí`/`no`):

| Operación        | size | capacity | ¿hubo resize? |
|-------------------|------|----------|----------------|
| `append(4)`       |      |          |                |
| `append(8)`       |      |          |                |
| `append(15)`      |      |          |                |
| `append(16)`      |      |          |                |
| `append(23)`      |      |          |                |
| `append(42)`      |      |          |                |
| `append(7)`       |      |          |                |

**Respuesta:**

| Operación        | size | capacity | ¿hubo resize? |
|-------------------|------|----------|----------------|
| `append(4)`       |      |          |                |
| `append(8)`       |      |          |                |
| `append(15)`      |      |          |                |
| `append(16)`      |      |          |                |
| `append(23)`      |      |          |                |
| `append(42)`      |      |          |                |
| `append(7)`       |      |          |                |

1. Número de redimensiones:

2. Costo amortizado por operación y justificación:

## Ejercicio 5 — Búsqueda recursiva en una lista

### Parte A — Búsqueda lineal recursiva (lista sin ordenar)

Implemente `busqueda_lineal_recursiva(lista, x)`, que retorna `True` si `x` está en `lista` y `False` en caso contrario. Debe ser **recursiva** (nada de `for`/`while`) y **no puede usar el operador `in`** ni métodos como `list.index`/`list.count`. Solo puede indexar la lista (`lista[i]`) y comparar elementos.

### Parte B — Búsqueda binaria recursiva (lista ordenada)

Ahora suponga que la lista está **ordenada** de menor a mayor. Implemente `busqueda_binaria_recursiva(lista, x)`, que también retorna `True`/`False`, pero de forma recursiva usando el algoritmo de **búsqueda binaria**: en cada llamada, compare `x` con el elemento de la mitad del rango actual y descarte la mitad donde `x` no puede estar.

Compare al final la cantidad de comparaciones que hace cada enfoque en el peor caso.

In [14]:
def busqueda_lineal_recursiva(lista, x, i=0):
    """Retorna True si x esta en lista[i:], de forma recursiva (sin 'in', sin bucles)."""
    if i == len(lista):
        return False

    # Encontramos x
    if lista[i] == x:
        return True

    # Continuamos buscando en la siguiente posicion
    return busqueda_lineal_recursiva(lista, x, i + 1)

In [15]:
desordenada = [8, 3, 15, 1, 9, 22, 4]

print(busqueda_lineal_recursiva(desordenada, 9))    # esperado: True
print(busqueda_lineal_recursiva(desordenada, 4))    # esperado: True
print(busqueda_lineal_recursiva(desordenada, 100))  # esperado: False
print(busqueda_lineal_recursiva([], 5))             # esperado: False (lista vacia)

True
True
False
False


In [16]:
def busqueda_binaria_recursiva(lista, x, izq=0, der=None):
    """Retorna True si x esta en lista (ordenada de menor a mayor), usando busqueda binaria recursiva."""
    if der is None:
        der = len(lista) - 1
    # Caso base: el rango ya no tiene elementos
    if izq > der:
        return False

    # Posicion del elemento central
    mitad = (izq + der) // 2

    # Encontramos x
    if lista[mitad] == x:
        return True

    # x es menor: buscamos en la mitad izquierda
    if x < lista[mitad]:
        return busqueda_binaria_recursiva(lista, x, izq, mitad - 1)

    # x es mayor: buscamos en la mitad derecha
    return busqueda_binaria_recursiva(lista, x, mitad + 1, der)

In [17]:
ordenada = [1, 3, 4, 8, 9, 15, 22]

print(busqueda_binaria_recursiva(ordenada, 9))    # esperado: True
print(busqueda_binaria_recursiva(ordenada, 1))    # esperado: True (extremo izquierdo)
print(busqueda_binaria_recursiva(ordenada, 22))   # esperado: True (extremo derecho)
print(busqueda_binaria_recursiva(ordenada, 100))  # esperado: False
print(busqueda_binaria_recursiva([], 5))          # esperado: False (lista vacia)

True
True
True
False
False


**Pregunta:** en el peor caso, ¿cuántas comparaciones hace `busqueda_lineal_recursiva` y cuántas `busqueda_binaria_recursiva` sobre una lista de `n` elementos? Exprese ambas en notación O-grande.

In [18]:
# Respuesta:
# busqueda_lineal_recursiva():  O(n)
# busqueda_binaria_recursiva(): O(log n)
# Justificacion:
#La busqueda lineal puede comparar los n elementos en el peor
# caso. La busqueda binaria descarta aproximadamente la mitad de los elementos
# en cada comparacion, por lo que necesita O(log n) comparaciones.


In [ ]:
Quiero estudiar y comprender estructuras de datos en Python, específicamente Stack (pila) y Queue (cola), y posteriormente entender una implementación de una Stack usando una sola ArrayQueue.

Mi objetivo NO es memorizar código, sino comprender visualmente qué ocurre con los elementos y por qué cada línea existe.

Conceptos que ya trabajamos:

1. Stack / Pila
- Funciona con LIFO: Last In, First Out.
- El último elemento que entra es el primero que sale.
- Conceptualmente:
  push() → agregar
  pop() → sacar
  top() → mirar el elemento superior sin sacarlo.

Ejemplo:
stack = []
stack.append("A")
stack.append("B")
stack.append("C")

stack.pop() → "C"

2. Queue / Cola
- Funciona con FIFO: First In, First Out.
- El primer elemento que entra es el primero que sale.
- Conceptualmente:
  enqueue() → agregar
  dequeue() → sacar
  first() → mirar el primero sin sacarlo.

En Python, deque puede trabajar por ambos extremos:
- append() → agrega al final/derecha.
- pop() → elimina del final/derecha.
- popleft() → elimina del principio/izquierda.

Importante: deque NO significa necesariamente FIFO. Es una estructura de doble extremo. Puede usarse para comportarse como Stack o Queue dependiendo de las operaciones:
- append() + pop() → comportamiento LIFO.
- append() + popleft() → comportamiento FIFO.

3. Ejercicio que quiero llegar a comprender:

class StackConCola:
    """Implementacion del ADT de pila usando una unica ArrayQueue como almacenamiento."""

    def __init__(self):
        self._datos = ArrayQueue()

    def __len__(self):
        return len(self._datos)

    def is_empty(self):
        return len(self._datos) == 0

    def push(self, e):
        """Agrega e a la cima de la pila."""
        self._datos.enqueue(e)

        for _ in range(len(self._datos) - 1):
            self._datos.enqueue(self._datos.dequeue())

    def top(self):
        if self.is_empty():
            raise Exception("Stack is empty")
        return self._datos.first()

    def pop(self):
        if self.is_empty():
            raise Empty("Stack is empty")
        return self._datos.dequeue()

    def __str__(self):
        return str(self._datos)

El ejercicio pide implementar el Stack ADT usando una sola ArrayQueue como variable de instancia (_datos), sin listas ni estructuras auxiliares. Dentro de cada método solo se permite memoria local constante.

La idea del push es:
- Primero hacer enqueue(e), por lo que el nuevo elemento queda al final.
- Después rotar todos los elementos que estaban antes que e.
- Cada rotación consiste en dequeue() del frente y enqueue() de ese mismo elemento al final.
- Después de las rotaciones, el nuevo elemento queda al frente.
- Así, dequeue() puede funcionar como pop() de la Stack.

Ejemplo:

Después de:
enqueue(A)
enqueue(B)
enqueue(C)
enqueue(D)

tenemos:

[A] [B] [C] [D]
 ↑
FRONT

Rotación 1:
[B] [C] [D] [A]

Rotación 2:
[C] [D] [A] [B]

Rotación 3:
[D] [A] [B] [C]

Ahora D está al frente, por lo que dequeue() devuelve D y obtenemos comportamiento LIFO.

También entendí que:

len(self._datos) - 1

NO significa necesariamente "no tomar el último índice". En este caso estamos calculando cuántas veces debe ejecutarse el for. Si hay 4 elementos después de insertar D, necesitamos 3 rotaciones porque solo debemos mover los 3 elementos que estaban antes de D.

Complejidades:
- push() → O(n)
- pop() → O(1)
- top() → O(1)

METODOLOGÍA QUE QUIERO:
- Practicar UNO POR UNO, no darme muchos ejercicios de golpe.
- Hacer preguntas de opción múltiple A/B/C/D.
- Usar dibujos ASCII de la estructura para visualizar los elementos.
- Si respondo mal, NO avanzar inmediatamente: explicar el error y volver a preguntarme algo parecido.
- Quiero razonar qué ocurre con cada operación, no memorizar.
- Empezar con ejercicios muy básicos de Stack y Queue y aumentar gradualmente la dificultad.
- Después pasar a ArrayQueue con enqueue/dequeue/first.
- Luego practicar rotaciones manualmente.
- Finalmente hacer preguntas específicas sobre StackConCola:
  * qué hace push()
  * por qué primero se hace enqueue(e)
  * qué hace el for
  * por qué len(self._datos) - 1
  * qué hace self._datos.dequeue()
  * qué hace self._datos.enqueue(...)
  * por qué top() usa first()
  * por qué pop() usa dequeue()
  * complejidad temporal de cada método
  * eventualmente pedirme completar o escribir la clase desde cero.

IMPORTANTE:
No quiero que me des la solución completa de inmediato. Quiero descubrirla mediante preguntas y ejercicios progresivos.
1